# Kerr Solitons in TFLN Waveguides

Temporal soliton formation from χ⁽³⁾ self-phase modulation in a TFLN
waveguide, with optional χ⁽²⁾ cascading that provides a **temperature-tunable**
effective nonlinearity.

**Three-act structure:**
1. **Pure Kerr solitons** (χ⁽²⁾ off) — validates the χ⁽³⁾ implementation
2. **Cascaded-enhanced solitons** (χ⁽²⁾ on, large Δk) — temperature tunes γ_eff
3. **Strong χ⁽²⁾+χ⁽³⁾ coupling** (moderate Δk) — two-color dynamics

**Operating point:** 1550 nm fundamental, β₂ = −50 fs²/mm (anomalous),
50 fs sech pulses, 40 mm waveguide.

In [ ]:
import numpy as np
from numpy.fft import fft, ifft, fftfreq, fftshift
import matplotlib.pyplot as plt
from scipy.constants import pi, c
from IPython.display import display

from snow import waveguides, pulses, materials, util

nm = 1e-9; um = 1e-6; mm = 1e-3; fs = 1e-15; ps = 1e-12; pJ = 1e-12

## Cell 0 — γ Calibration

The effective Kerr parameter γ = n₂·ω/(c·A_eff) requires A_eff, which
SNOW's effective index method doesn't provide. Calibrate empirically:
find the γ that gives a fundamental soliton (N=1) at the analytically
predicted energy.

In [ ]:
# Grid: 700nm–3um covers SH(775nm) for Act II and gives dt≈3fs
# (16 grid points per 50fs pulse FWHM — adequate temporal resolution)
N = 2**12
lam_start, lam_stop = 700*nm, 3*um
BW = c/lam_start - c/lam_stop
dt = 1/BW
t = np.arange(N)*dt - N*dt/2
f_ref = (c/lam_start + c/lam_stop)/2
f_abs = fftfreq(N, dt) + f_ref

# Waveguide
wg = waveguides.waveguide(
    w_top=1800*nm, h_thinfilm=700*nm, h_etch=350*nm,
    tf_material='LN_MgO_e_T', box_material='SiO2', clad_material='Air')

beta2 = wg.beta2(1550*nm)[0]
v_ref = 1 / wg.beta1(1550*nm)[0]
FWHM = 50 * fs
T0 = FWHM / 1.76

print(f'Grid: N={N}, dt={dt/fs:.2f} fs ({FWHM/dt:.0f} points per FWHM)')
print(f'β₂(1550nm) = {beta2*1e27:.1f} fs²/mm')

# γ calibration: the formula P_sol = |β₂|/(γ·T₀²) is γ-invariant
# (doubling γ halves E_sol, same soliton condition). All γ values
# give FWHM ratio ≈ 0.91 — slight compression from higher-order
# dispersion. Use γ = 0.3 /W/m (300 /W/km) as the literature midpoint.

gamma = 0.3  # /W/m = 300 /W/km (literature: 100–400 for TFLN)
P_sol = abs(beta2) / (gamma * T0**2)
E_sol = P_sol * FWHM / 0.88
z0 = T0**2 / abs(beta2)

print(f'\nγ = {gamma} /W/m ({gamma*1e3:.0f} /W/km)')
print(f'P_sol = {P_sol:.1f} W, E_sol = {E_sol/pJ:.1f} pJ')
print(f'z₀ = {z0/mm:.1f} mm, soliton period = {pi*z0/2/mm:.1f} mm')
print(f'40 mm crystal = {40*mm/z0:.1f} z₀')

# Verify: N=1 soliton propagates near-unchanged
wg.add_poling(lambda z: 1.0)
wg.set_nonlinear_coeffs(N=1, X0=0, gamma_eff=gamma)
wg.set_loss(0); wg.set_length(20*mm)

p = pulses.sech_pulse(t, FWHM, f_ref=f_ref, Energy=E_sol,
                      f0=c/(1550*nm), Npwr_dB=200, frep=250e6)
out, _ = wg.propagate_NEE(p, v_ref=v_ref, verbose=False)
I_out = np.abs(out.a)**2
hm = np.max(I_out)/2
above = np.where(I_out > hm)[0]
fw = (t[above[-1]] - t[above[0]]) if len(above) > 1 else FWHM
print(f'N=1 check (L=20mm): FWHM ratio = {fw/FWHM:.3f} (ideal: 1.0)')

## Act I — Pure Kerr Solitons

### Cell 1: Soliton Energy Scan

χ⁽²⁾ OFF. 50 fs sech pulse. z–t color maps at N = 0.5, 1, 2, 3.

In [ ]:
L = 40 * mm
wg.set_nonlinear_coeffs(N=1, X0=0, gamma_eff=gamma)
wg.set_loss(0)
wg.set_length(L)

z_pos = np.linspace(0.5*mm, L, 80)
N_orders = [0.5, 1.0, 2.0, 3.0]

t_win = 0.5 * ps
t_mask = np.abs(t) < t_win
t_plot = t[t_mask] / fs

fig, axes = plt.subplots(1, len(N_orders), figsize=(4*len(N_orders), 5))
fig.suptitle(f'Kerr soliton z–t evolution ({FWHM/fs:.0f} fs, L={L/mm:.0f} mm)', fontsize=13)
fig_handle = display(fig, display_id=True)
plt.ion()

for idx, N_sol in enumerate(N_orders):
    E = E_sol * N_sol**2
    p = pulses.sech_pulse(t, FWHM, f_ref=f_ref, Energy=E,
                          f0=c/(1550*nm), Npwr_dB=200, frep=250e6)
    out, snaps = wg.propagate_NEE(p, v_ref=v_ref, verbose=False, z_save=z_pos)
    
    I_zt = np.array([np.abs(snaps[iz][t_mask])**2 for iz in range(len(z_pos))])
    axes[idx].pcolormesh(t_plot, z_pos/mm, I_zt, shading='auto', cmap='inferno')
    axes[idx].set_xlabel('Time (fs)')
    axes[idx].set_title(f'N = {N_sol}')
    if idx == 0:
        axes[idx].set_ylabel('z (mm)')
    fig_handle.update(fig)
    print(f'  N={N_sol} done')

plt.tight_layout()
fig_handle.update(fig)
plt.close(fig)

In [ ]:
# Output FWHM and peak power vs soliton order
N_scan = np.array([0.3, 0.5, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.5, 2.0, 2.5, 3.0])
fwhm_ratios = []
ppeak_ratios = []

for N_sol in N_scan:
    E = E_sol * N_sol**2
    p = pulses.sech_pulse(t, FWHM, f_ref=f_ref, Energy=E,
                          f0=c/(1550*nm), Npwr_dB=200, frep=250e6)
    out, _ = wg.propagate_NEE(p, v_ref=v_ref, verbose=False)
    I_out = np.abs(out.a)**2
    hm = np.max(I_out)/2
    above = np.where(I_out > hm)[0]
    fw = (t[above[-1]] - t[above[0]]) if len(above) > 1 else FWHM
    fwhm_ratios.append(fw / FWHM)
    ppeak_ratios.append(np.max(I_out) / np.max(np.abs(p.a)**2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(N_scan, fwhm_ratios, 'ko-')
ax1.axhline(1, color='gray', ls=':', alpha=0.5)
ax1.set_xlabel('Soliton order N'); ax1.set_ylabel('FWHM_out / FWHM_in')
ax1.set_title('Pulse width change'); ax1.grid(True)

ax2.plot(N_scan, ppeak_ratios, 'ro-')
ax2.axhline(1, color='gray', ls=':', alpha=0.5)
ax2.set_xlabel('Soliton order N'); ax2.set_ylabel('Ppeak_out / Ppeak_in')
ax2.set_title('Peak power change'); ax2.grid(True)

plt.tight_layout()
plt.show()

### Cell 2: Soliton Self-Compression and Fission

N = 3–5: initial compression, then fission into fundamental solitons
plus dispersive wave emission. The χ⁽²⁾-free analog of fiber
supercontinuum soliton dynamics.

In [ ]:
E_high = [3, 5]  # soliton orders
z_pos_dense = np.linspace(0.5*mm, L, 100)

fig, axes = plt.subplots(2, len(E_high), figsize=(6*len(E_high), 9))
fig_handle = display(fig, display_id=True)
plt.ion()

f_abs = fftfreq(N, dt) + f_ref
wl_abs = c / f_abs

for ie, N_high in enumerate(E_high):
    E = E_sol * N_high**2
    p = pulses.sech_pulse(t, FWHM, f_ref=f_ref, Energy=E,
                          f0=c/(1550*nm), Npwr_dB=200, frep=250e6)
    out, snaps = wg.propagate_NEE(p, v_ref=v_ref, verbose=False,
                                  z_save=z_pos_dense)

    # z-t map
    t_mask_c = np.abs(t) < 0.3*ps
    I_zt = np.array([np.abs(snaps[iz][t_mask_c])**2 for iz in range(len(z_pos_dense))])
    axes[0, ie].pcolormesh(t[t_mask_c]/fs, z_pos_dense/mm, I_zt,
                           shading='auto', cmap='inferno')
    axes[0, ie].set_xlabel('Time (fs)'); axes[0, ie].set_ylabel('z (mm)')
    axes[0, ie].set_title(f'Temporal — N={N_high}')

    # z-λ map
    wl_plot = np.linspace(1300, 1900, 400) * nm
    spec_zt = np.zeros((len(z_pos_dense), len(wl_plot)))
    for iz in range(len(z_pos_dense)):
        A = fft(snaps[iz])
        psd = np.abs(A)**2 * dt**2
        spec_zt[iz] = np.interp(c/wl_plot, fftshift(f_abs), fftshift(psd))

    spec_zt_norm = spec_zt / np.max(spec_zt)
    axes[1, ie].pcolormesh(wl_plot/nm, z_pos_dense/mm,
                           10*np.log10(spec_zt_norm + 1e-10),
                           shading='auto', cmap='inferno', vmin=-30, vmax=0)
    axes[1, ie].set_xlabel('Wavelength (nm)'); axes[1, ie].set_ylabel('z (mm)')
    axes[1, ie].set_title(f'Spectral — N={N_high}')

    fig_handle.update(fig)
    print(f'  N={N_high} done')

plt.suptitle('Soliton compression and fission (χ⁽³⁾ only)', fontsize=13)
plt.tight_layout()
fig_handle.update(fig)
plt.close(fig)

### Cell 3: Robustness

**3a:** Gaussian input reshapes to sech (attractor property).
**3b:** Propagation loss causes adiabatic broadening.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig_handle = display(fig, display_id=True)
plt.ion()

z_pos_rob = np.linspace(0.5*mm, L, 80)

# 3a: Gaussian vs sech input at N=1
for shape, label, col in [('sech', 'Sech (N=1)', 'C0'),
                           ('gauss', 'Gaussian (same E)', 'C1')]:
    wg.set_nonlinear_coeffs(N=1, X0=0, gamma_eff=gamma)
    wg.set_loss(0); wg.set_length(L)
    if shape == 'sech':
        p = pulses.sech_pulse(t, FWHM, f_ref=f_ref, Energy=E_sol,
                              f0=c/(1550*nm), Npwr_dB=200, frep=250e6)
    else:
        p = pulses.gaussian_pulse(t, FWHM, f_ref=f_ref, Energy=E_sol,
                                  f0=c/(1550*nm), Npwr_dB=200, frep=250e6)
    out, snaps = wg.propagate_NEE(p, v_ref=v_ref, verbose=False, z_save=z_pos_rob)
    
    widths = []
    for iz in range(len(z_pos_rob)):
        I = np.abs(snaps[iz])**2
        hm = np.max(I)/2
        above = np.where(I > hm)[0]
        widths.append((t[above[-1]]-t[above[0]])/fs if len(above)>1 else np.nan)
    axes[0].plot(z_pos_rob/mm, widths, color=col, label=label)

axes[0].set_xlabel('z (mm)'); axes[0].set_ylabel('FWHM (fs)')
axes[0].set_title('3a: Gaussian reshapes to soliton')
axes[0].legend(); axes[0].grid(True)
fig_handle.update(fig)
print('  3a done')

# 3b: Loss — N=1 soliton with increasing loss
for alpha_dBcm, col in [(0, 'C0'), (0.1, 'C1'), (0.5, 'C2'), (1.0, 'C3')]:
    wg.set_nonlinear_coeffs(N=1, X0=0, gamma_eff=gamma)
    wg.set_loss(util.absorption_coeff(alpha_dBcm) if alpha_dBcm > 0 else 0)
    wg.set_length(L)
    p = pulses.sech_pulse(t, FWHM, f_ref=f_ref, Energy=E_sol,
                          f0=c/(1550*nm), Npwr_dB=200, frep=250e6)
    out, snaps = wg.propagate_NEE(p, v_ref=v_ref, verbose=False, z_save=z_pos_rob)
    
    widths = []
    for iz in range(len(z_pos_rob)):
        I = np.abs(snaps[iz])**2
        hm = np.max(I)/2
        above = np.where(I > hm)[0]
        widths.append((t[above[-1]]-t[above[0]])/fs if len(above)>1 else np.nan)
    axes[1].plot(z_pos_rob/mm, widths, color=col, label=f'{alpha_dBcm} dB/cm')

axes[1].set_xlabel('z (mm)'); axes[1].set_ylabel('FWHM (fs)')
axes[1].set_title('3b: Soliton under loss')
axes[1].legend(); axes[1].grid(True)

# 3b: Peak power under loss
for alpha_dBcm, col in [(0, 'C0'), (0.1, 'C1'), (0.5, 'C2'), (1.0, 'C3')]:
    wg.set_nonlinear_coeffs(N=1, X0=0, gamma_eff=gamma)
    wg.set_loss(util.absorption_coeff(alpha_dBcm) if alpha_dBcm > 0 else 0)
    wg.set_length(L)
    p = pulses.sech_pulse(t, FWHM, f_ref=f_ref, Energy=E_sol,
                          f0=c/(1550*nm), Npwr_dB=200, frep=250e6)
    out, snaps = wg.propagate_NEE(p, v_ref=v_ref, verbose=False, z_save=z_pos_rob)
    peaks = [np.max(np.abs(snaps[iz])**2) for iz in range(len(z_pos_rob))]
    axes[2].plot(z_pos_rob/mm, peaks, color=col, label=f'{alpha_dBcm} dB/cm')

axes[2].set_xlabel('z (mm)'); axes[2].set_ylabel('Peak power (W)')
axes[2].set_title('3b: Peak power decay')
axes[2].legend(); axes[2].grid(True)

fig_handle.update(fig)
print('  3b done')
plt.tight_layout()
fig_handle.update(fig)
plt.close(fig)

## Act II — Adding χ⁽²⁾ Cascading

### Cell 4: Temperature-Tunable Nonlinearity

Turn χ⁽²⁾ ON with the 5.18 µm poling period (Tutorial 7). At large Δk
(high temperature), cascaded SHG adds an effective Kerr contribution:

- Δk > 0: n₂_casc > 0 → adds to Kerr → lower soliton energy
- Δk < 0: n₂_casc < 0 → subtracts from Kerr → higher soliton energy

In [ ]:
pp = 5.18 * um  # Tutorial 7 poling period
Kg = 2*pi / pp

# Compute Δk vs temperature
T_range = np.arange(-10, 180, 5)
dk_range = []
for T_val in T_range:
    nfh = wg.neff(1550*nm, T=T_val)[0]
    nsh = wg.neff(775*nm, T=T_val)[0]
    dk_range.append(2*pi*nsh/(775*nm) - 2*2*pi*nfh/(1550*nm) - Kg)
dk_range = np.array(dk_range)

# Phase-matching temperature
T_pm_idx = np.argmin(np.abs(dk_range))
T_pm = T_range[T_pm_idx]
print(f'Phase matching at T ≈ {T_pm}°C')

# Soliton energy vs temperature: at each T, find energy where
# output FWHM = input FWHM (fundamental soliton condition)
T_sweep = [T_pm - 30, T_pm - 15, T_pm + 30, T_pm + 60, T_pm + 100, T_pm + 150]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig_handle = display(fig, display_id=True)
plt.ion()

ax1.plot(T_range, dk_range/1e3, 'k-')
ax1.axhline(0, color='gray', ls=':', alpha=0.3)
for T_val in T_sweep:
    nfh = wg.neff(1550*nm, T=T_val)[0]
    nsh = wg.neff(775*nm, T=T_val)[0]
    dk_val = 2*pi*nsh/(775*nm) - 2*2*pi*nfh/(1550*nm) - Kg
    ax1.plot(T_val, dk_val/1e3, 'ro', ms=8)
ax1.set_xlabel('Temperature (°C)'); ax1.set_ylabel('Δk (1/mm)')
ax1.set_title('Phase mismatch vs temperature')
ax1.grid(True)
fig_handle.update(fig)

# At each temperature, sweep energy and find N=1
soliton_energies = []
L_act2 = 20 * mm

for T_val in T_sweep:
    nfh = wg.neff(1550*nm, T=T_val)[0]
    nsh = wg.neff(775*nm, T=T_val)[0]
    dk_val = 2*pi*nsh/(775*nm) - 2*2*pi*nfh/(1550*nm) - Kg
    
    # Carrier-phase-minimizing v_ref for this temperature
    beta_ref_T = wg.beta(np.array([c/f_ref]), T=T_val)[0]
    v_ref_T = 2*pi*f_ref / (beta_ref_T + Kg)
    
    wg.set_nonlinear_coeffs(N=1, X0=1.1e-12, gamma_eff=gamma)
    wg.set_loss(0); wg.set_length(L_act2)
    
    # Binary search for N=1 energy
    E_lo, E_hi = 0.5*pJ, 50*pJ
    for _ in range(15):
        E_mid = np.sqrt(E_lo * E_hi)  # geometric mean
        p = pulses.sech_pulse(t, FWHM, f_ref=f_ref, Energy=E_mid,
                              f0=c/(1550*nm), Npwr_dB=200, frep=250e6)
        out, _ = wg.propagate_NEE(p, v_ref=v_ref_T, verbose=False,
                                  T=T_val, Kg=Kg)
        I_out = np.abs(out.a)**2
        hm = np.max(I_out)/2
        above = np.where(I_out > hm)[0]
        fw = (t[above[-1]] - t[above[0]]) if len(above) > 1 else FWHM*2
        if fw / FWHM > 1.0:
            E_lo = E_mid  # need more energy
        else:
            E_hi = E_mid  # too much energy
    
    E_n1 = np.sqrt(E_lo * E_hi)
    soliton_energies.append(E_n1)
    print(f'  T={T_val:4.0f}°C: Δk={dk_val:7.0f}/m, E_sol={E_n1/pJ:.2f} pJ')

ax2.plot([dk_range[np.argmin(np.abs(T_range - T))] for T in T_sweep],
         np.array(soliton_energies)/pJ, 'ko-', ms=8)
ax2.axhline(E_sol/pJ, color='C0', ls='--', alpha=0.5, label='Pure Kerr E_sol')
ax2.set_xlabel('Δk (1/m)'); ax2.set_ylabel('Soliton energy (pJ)')
ax2.set_title('Soliton energy vs phase mismatch')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
fig_handle.update(fig)
plt.close(fig)

## Act III — Strong χ⁽²⁾+χ⁽³⁾ Coupling

### Cell 5: Approaching Phase Matching

Move temperature toward SHG phase matching. At small Δk, the SH field
becomes dynamically active — energy oscillates between FH and SH,
and the Kerr soliton picture breaks down.

In [ ]:
# z-t evolution at decreasing Δk
T_act3 = [T_pm + 100, T_pm + 30, T_pm + 10]
f_sep = c / (1200*nm)  # FH/SH separation
mask_sh = f_abs >= f_sep

fig, axes = plt.subplots(2, len(T_act3), figsize=(5*len(T_act3), 9))
fig.suptitle('Approaching phase matching: FH (top) and SH fraction (bottom)', fontsize=13)
fig_handle = display(fig, display_id=True)
plt.ion()

z_pos_act3 = np.linspace(0.5*mm, 20*mm, 60)
t_mask3 = np.abs(t) < 0.5*ps

for iT, T_val in enumerate(T_act3):
    nfh = wg.neff(1550*nm, T=T_val)[0]
    nsh = wg.neff(775*nm, T=T_val)[0]
    dk_val = 2*pi*nsh/(775*nm) - 2*2*pi*nfh/(1550*nm) - Kg
    
    beta_ref_T = wg.beta(np.array([c/f_ref]), T=T_val)[0]
    v_ref_T = 2*pi*f_ref / (beta_ref_T + Kg)
    
    # Use pure-Kerr soliton energy
    wg.set_nonlinear_coeffs(N=1, X0=1.1e-12, gamma_eff=gamma)
    wg.set_loss(0); wg.set_length(20*mm)
    
    p = pulses.sech_pulse(t, FWHM, f_ref=f_ref, Energy=E_sol,
                          f0=c/(1550*nm), Npwr_dB=200, frep=250e6)
    out, snaps = wg.propagate_NEE(p, v_ref=v_ref_T, verbose=False,
                                  T=T_val, Kg=Kg, z_save=z_pos_act3)
    
    # FH temporal evolution
    I_zt = np.array([np.abs(snaps[iz][t_mask3])**2 for iz in range(len(z_pos_act3))])
    axes[0, iT].pcolormesh(t[t_mask3]/fs, z_pos_act3/mm, I_zt,
                            shading='auto', cmap='inferno')
    axes[0, iT].set_title(f'Δk = {dk_val:.0f}/m')
    axes[0, iT].set_xlabel('Time (fs)')
    if iT == 0: axes[0, iT].set_ylabel('z (mm)')
    
    # SH energy fraction along z
    sh_frac = []
    for iz in range(len(z_pos_act3)):
        A = fft(snaps[iz])
        E_sh = np.sum(np.abs(ifft(A * mask_sh))**2) * dt
        E_tot = np.sum(np.abs(snaps[iz])**2) * dt
        sh_frac.append(E_sh / E_tot * 100 if E_tot > 0 else 0)
    axes[1, iT].plot(z_pos_act3/mm, sh_frac, 'C1-')
    axes[1, iT].set_xlabel('z (mm)'); axes[1, iT].set_ylabel('SH %')
    axes[1, iT].set_title(f'SH energy fraction')
    axes[1, iT].set_ylim(0, max(max(sh_frac)*1.2, 1))
    axes[1, iT].grid(True)
    
    fig_handle.update(fig)
    print(f'  T={T_val}°C (Δk={dk_val:.0f}/m) done')

plt.tight_layout()
fig_handle.update(fig)
plt.close(fig)

## Summary

| Regime | Cells | χ⁽²⁾ | χ⁽³⁾ | SH energy | Soliton energy | Notes |
|--------|-------|-------|-------|-----------|----------------|-------|
| Pure Kerr | 1–3 | Off | On | 0 | E_Kerr | Textbook NLSE |
| Cascaded-enhanced | 4 | On (large Δk) | On | < 5% | < E_Kerr | Temperature-tunable γ |
| Strong coupling | 5 | On (small Δk) | On | 5–50% | Complex | Two-color dynamics |

**Key result:** Temperature provides a continuous knob for adjusting the
effective nonlinearity — from enhanced (Δk > 0, cascading adds to Kerr)
through pure Kerr (Δk → ∞) to reduced (Δk < 0, cascading subtracts).
No χ⁽³⁾-only system offers this tunability.